In [2]:
import argparse
import os
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [ ]:
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import torchvision.transforms.functional as RF
from PIL import Image
import numpy as np
import random,cv2,pandas,os,numpy
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [ ]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

nz = 100
beta1 = 0.5
lr = 0.0001
batch_size=100000
ngpu=1
ngf,nc = 3,3
ndf = 64

device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

In [ ]:
x = pandas.read_csv("/kaggle/input/rohlik-sales-forecasting-challenge-v2/sales_train.csv").sort_values(by=['date', 'warehouse'])

x = torch.from_numpy(numpy.nan_to_num(x[['total_orders', 'sell_price_main', 'type_0_discount', 'type_1_discount',
                                         'type_2_discount', 'type_3_discount', 'type_4_discount',  'type_5_discount', 
                                         'type_6_discount']].to_numpy().astype(numpy.float32)).reshape((13, 308263, 9)))
x.shape #4315682

class EC_NET(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.rafire = nn.Sequential(       
            nn.Linear(9, 1524),
            nn.BatchNorm1d(1524),
            nn.LeakyReLU(),
            
            nn.Linear(1524, 824),
            nn.BatchNorm1d(824),
            nn.LeakyReLU(),
            
            nn.Linear(824, 424),
            nn.BatchNorm1d(424),
            nn.LeakyReLU(),
            
            nn.Linear(424, 824),
            nn.BatchNorm1d(824),
            nn.LeakyReLU(),
            
            nn.Linear(824, 1524),
            nn.BatchNorm1d(1524),
            nn.LeakyReLU(),

            nn.Linear(1524, 100)
        )

    def forward(self,x):
        
        return self.rafire(x)
        
ec_net = EC_NET().type(torch.float32).to(device).eval()
ec_net= nn.DataParallel(ec_net).to(device)
#ec_net.load_state_dict(torch.load("/kaggle/input/encoder-weight-data/weight.pth",weights_only=False,map_location=torch.device('cpu')))

j=0
for i in x:
    print(f"loding batch : {j}")
    if j==0:
        encode = ec_net(i).cpu().detach().numpy().reshape((len(i),3,10,10))
        #print(encode[0])
        plt.imshow(encode[0][0])
        plt.show()
    else:
        encode = numpy.concatenate((encode, ec_net(i).cpu().detach().numpy().reshape((len(i),3,10,10))), axis=0)
        plt.imshow(encode[0][0])
        plt.show()
    #if j==25:
    #    break
    j+=1

x=torch.Tensor(encode)

In [ ]:
x.shape

In [ ]:
self.rafire = nn.Sequential(    
            nn.Linear(9, 24),
            nn.BatchNorm1d(24),
            nn.LeakyReLU(),

            nn.Linear(24, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(),

            nn.Linear(64, 124),
            nn.BatchNorm1d(124),
            nn.LeakyReLU(),

            nn.Linear(124, 264),
            nn.BatchNorm1d(264),
            nn.LeakyReLU(),

            nn.Linear(264, 424),
            nn.BatchNorm1d(424),
            nn.LeakyReLU(),

            nn.Linear(424, 864),
            nn.BatchNorm1d(864),
            nn.LeakyReLU(),

            nn.Linear(864, 1024),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(),
            
            nn.Linear(1024, 2024),
            nn.BatchNorm1d(2024),
            nn.LeakyReLU(),

            nn.Linear(2024, 4024),
            nn.BatchNorm1d(4024),
            nn.LeakyReLU(),
            
            nn.Linear(4024, 8024),
            nn.BatchNorm1d(8024),
            nn.LeakyReLU(),

            nn.Linear(8024, 16024),
            nn.BatchNorm1d(16024),
            nn.LeakyReLU(),
            
            nn.Linear(16024, 32024),
            nn.BatchNorm1d(32024),
            nn.LeakyReLU(),
            
            nn.Linear(32024, 30000)
        )